In [2]:

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
import faiss

import warnings
warnings.filterwarnings("ignore")

In [3]:
# Load TXT Files
loader = DirectoryLoader(
    "data/",
    glob="*.txt",
    loader_cls=TextLoader
)

documents = loader.load()

print(f"Loaded {len(documents)} documents")


Loaded 2 documents


In [13]:
# Split Documents

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

split_docs = splitter.split_documents(documents)

for i, doc in enumerate(split_docs):
    doc.metadata["chunk_id"] = i

print(f"Split into {len(split_docs)} chunks")


Split into 18 chunks


In [14]:
# Embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Auto Detect Embedding Dimension
embedding_dim = len(
    embeddings.embed_query("hello")
)

print("Embedding Dimension:", embedding_dim)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2305.29it/s]


Embedding Dimension: 384


In [15]:
# FAISS Index

index = faiss.IndexFlatL2(embedding_dim)

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

vector_store.add_documents(split_docs)

print("Documents indexed")

Documents indexed


In [16]:
# Retriever

retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}
)

In [17]:
# LLM

llm = ChatOllama(model="llama3.2")

In [24]:
# Query

query = "What is the GDP of USA in 2025?"

In [25]:
# Retrieve Docs

retrieved_docs = retriever.invoke(query)

In [26]:
# Build Context

context_text = "\n\n".join(
    [doc.page_content for doc in retrieved_docs]
)

In [27]:
# Prompt

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Use ONLY the provided context to answer.
If not found, say "I don't know."

Context:
{context}

Question:
{question}

Answer:
""")


formatted_prompt = prompt.format(
    context=context_text,
    question=query
)

In [28]:
# LLM Response

response = llm.invoke(formatted_prompt)

print("\nAnswer:\n")
print(response.content)


Answer:

The United States' nominal Gross Domestic Product (GDP) in 2025 is estimated to be in the range of $29–$30 trillion USD.


In [29]:

# Show Sources
print("\nRetrieved Sources:\n")

for i, doc in enumerate(retrieved_docs, start=1):

    print(f"--- Result {i} ---")

    print("File:",
          doc.metadata.get("source", "N/A"))

    print("Chunk ID:",
          doc.metadata.get("chunk_id", "N/A"))

    print(doc.page_content[:300])

    print("\n")


Retrieved Sources:

--- Result 1 ---
File: data\usa2.txt
Chunk ID: 14
ðŸ‡ºðŸ‡¸ U.S. GDP in 2025 â€” Economic Overview
Nominal GDP Estimate (2025)

As of 2025, the United Statesâ€™ nominal Gross Domestic Product (GDP) is estimated to be in the range of $29â€“30 trillion USD, maintaining its position as the largest economy in the world. This growth is driven by strong c


--- Result 2 ---
File: data\usa.txt
Chunk ID: 1
U.S. GDP â€“ Size, Composition, and Global Share
As of 2024, the United Statesâ€™ nominal GDP is estimated to be around $28 trillion USD, accounting for approximately 25% of the global economy. It ranks #1 in the world by nominal GDP, far ahead of China (which ranks 2nd). The U.S. GDP per capita is 


--- Result 3 ---
File: data\usa2.txt
Chunk ID: 16
Role of Technology and Innovation

Technology remains a primary growth engine in 2025. Investments in artificial intelligence, semiconductors, cloud computing, and biotechnology continue to drive productivity gains and global